In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
        # print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#cell1 %% [code]
import subprocess, sys, os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(f"找不到 {pattern}，请检查是否在右侧 Add Data 挂载了对应的依赖库！")

# 100% 照抄原代码：寻找并安装 onnxruntime

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")
else:
    print('没有！')



# 100% 照抄原代码：寻找并降级/升级 TF 2.20 以解决死锁
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)
    print("✅ TF 2.20 installed from wheel")
except Exception as e:
    print(e)

# 导入所有后续需要的包
import onnxruntime as ort
import pandas as pd
import numpy as np
import librosa
from tqdm.auto import tqdm
import re

print("环境配置严格执行完毕！")

ONNX Runtime installed
✅ TF 2.20 installed from wheel
环境配置严格执行完毕！


In [3]:
# %% [code]
# ── Cell 2: 全局需求与配置 ──
from pathlib import Path
import onnxruntime as ort

AUDIO_DIR = Path("/kaggle/input/competitions/birdclef-2026/train_soundscapes")
OUTPUT_DIR = Path("/kaggle/working/extracted_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

onnx_hits = list(Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026").rglob("perch_v2.onnx"))
if not onnx_hits:
    raise FileNotFoundError("❌ 未找到 perch_v2.onnx")
ONNX_MODEL_PATH = onnx_hits[0]
print(f"✅ 模型路径: {ONNX_MODEL_PATH}")

SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_LEN_SEC = 60
FILE_SAMPLES = SR * FILE_LEN_SEC
N_WINDOWS = FILE_LEN_SEC // WINDOW_SEC

BATCH_FILES = 16
NUM_WORKERS = 4

# ==========================================
# 🚀 内存安全控制：每 1000 个文件清洗一次内存，但全部追加到同一个文件
# ==========================================
CHUNK_SIZE = 1000
# 这次不写 TARGET_GROUPS 了，直接默认跑全量 10000 个！

✅ 模型路径: /kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx


In [4]:
# %% [code]
# cell3
import pandas as pd
from pathlib import Path
import os

print("正在构建精确的 5 秒切片任务清单...")

# ====================================================
# 第一部分：处理 train-audio 的最佳切片 (Argmax)
# ====================================================
df_audio = pd.read_csv("/kaggle/input/datasets/waozhang/bc26train-audio/train_audio_argmax_segments.csv")

# 整理所需列
df_audio["filename"] = df_audio["filepath"].apply(lambda x: Path(x).name)
df_audio["start_sec"] = df_audio["best_start_sec"].astype(float)
df_audio["end_sec"] = df_audio["best_end_sec"].astype(float)
df_audio["source"] = "train-audio"
# 只保留我们需要的列
df_audio = df_audio[["filepath", "filename", "start_sec", "end_sec", "primary_label", "source"]]

# ====================================================
# 第二部分：处理 train_soundscapes 的人工标签切片
# ====================================================
df_sc = pd.read_csv("/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv")

# 拼接绝对路径
sc_base_dir = "/kaggle/input/competitions/birdclef-2026/train_soundscapes/"
df_sc["filepath"] = sc_base_dir + df_sc["filename"].astype(str)

# 确保时间列为数字类型 (有时官方给的是 "0:05" 这种字符串格式，这里做一个安全兼容)
def parse_time_to_sec(x):
    if isinstance(x, str) and ":" in x:
        return pd.to_timedelta(x).total_seconds()
    return float(x)

df_sc["start_sec"] = df_sc["start"].apply(parse_time_to_sec)
df_sc["end_sec"] = df_sc["end"].apply(parse_time_to_sec)
df_sc["source"] = "train_soundscapes"
# 只保留需要的列
df_sc = df_sc[["filepath", "filename", "start_sec", "end_sec", "primary_label", "source"]]

# ====================================================
# 合并为总任务表
# ====================================================
df_all_tasks = pd.concat([df_audio, df_sc], ignore_index=True)

# 转换为字典列表，方便在多线程中极速迭代
tasks_list = df_all_tasks.to_dict('records')

print(f"✅ 数据整合完毕！")
print(f"   -> Train Audio 提取了: {len(df_audio)} 个切片")
print(f"   -> Soundscapes 提取了: {len(df_sc)} 个切片")
print(f"   -> 总计待处理任务: {len(tasks_list)} 个切片")

正在构建精确的 5 秒切片任务清单...
✅ 数据整合完毕！
   -> Train Audio 提取了: 33023 个切片
   -> Soundscapes 提取了: 1478 个切片
   -> 总计待处理任务: 34501 个切片


In [5]:
# %% [code]
# cell4
import librosa
import soundfile as sf
import numpy as np
import onnxruntime as ort

SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC

# 1. 开启 ONNX 极致优化
so = ort.SessionOptions()
so.intra_op_num_threads = 4
so.inter_op_num_threads = 1
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# ONNX_MODEL_PATH 从上文的 Cell 2 继承
ort_session = ort.InferenceSession(str(ONNX_MODEL_PATH), sess_options=so, providers=["CPUExecutionProvider"])
input_name = ort_session.get_inputs()[0].name
output_names = [o.name for o in ort_session.get_outputs()]
emb_output_name = next(name for name in output_names if "embedding" in name.lower())
print("✅ ONNX 模型就绪！")

# 2. 精确 5 秒提取函数
def extract_5s_segment(task_dict):
    """根据任务字典中的时间戳，精确切割出对应的 160000 个采样点"""
    filepath = task_dict["filepath"]
    start_sec = task_dict["start_sec"]
    end_sec = task_dict["end_sec"]
    
    try:
        # 读取整段音频（因为部分音频也就十几秒，全读进内存切片比跳跃读取更快）
        y, sr = sf.read(str(filepath), dtype="float32", always_2d=False)
        if y.ndim == 2: y = y.mean(axis=1) # 转单声道
        if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
        
        # 核心：根据频率计算切割的起止索引
        start_frame = int(start_sec * SR)
        end_frame = int(end_sec * SR)
        y_slice = y[start_frame:end_frame]
        
        # 强制填充或截断到严格的 160000 长度 (Perch 输入刚需)
        if len(y_slice) < WINDOW_SAMPLES:
            y_slice = np.pad(y_slice, (0, WINDOW_SAMPLES - len(y_slice)))
        else:
            y_slice = y_slice[:WINDOW_SAMPLES]
            
        return y_slice
    except Exception as e:
        # 万一文件损坏，返回空静音片段防止程序崩溃
        return np.zeros(WINDOW_SAMPLES, dtype=np.float32)

✅ ONNX 模型就绪！


In [6]:
# %% [code]
# cell5
import time
import gc
import concurrent.futures
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# 每次送进模型的切片数量（我们切片是零散的，所以 Batch 可以设大一点）
BATCH_SIZE = 64
NUM_WORKERS = 4
OUTPUT_DIR = Path("/kaggle/working/extracted_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 最终输出路径
final_output_file = OUTPUT_DIR / "teacher_features_label.parquet"
parquet_writer = None

global_t0 = time.time()
print(f"🔥 开始流式提取并写入...")

# 异步线程池启动！
with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    
    # 派发第一个 Batch
    next_batch_tasks = tasks_list[0:BATCH_SIZE]
    future_audio =[executor.submit(extract_5s_segment, t) for t in next_batch_tasks]
    
    # 按照 Batch Size 推进
    for i in tqdm(range(0, len(tasks_list), BATCH_SIZE), desc="特征提取进度"):
        current_batch_tasks = next_batch_tasks
        
        # 1. 拿到当前切片的波形数据
        batch_audio = [f.result() for f in future_audio]
        
        # 2. 无缝派发下个 Batch 的读取任务
        next_start = i + BATCH_SIZE
        if next_start < len(tasks_list):
            next_batch_tasks = tasks_list[next_start : next_start + BATCH_SIZE]
            future_audio =[executor.submit(extract_5s_segment, t) for t in next_batch_tasks]
            
        # 3. 组装张量送入 ONNX (Shape: Batch, 160000)
        x_input = np.vstack(batch_audio).astype(np.float32)
        outputs = ort_session.run([emb_output_name], {input_name: x_input})
        embs = outputs[0]  # Shape: (Batch, 1536)
        
        # 4. 构建当前 Batch 的 DataFrame（严格按照你的列名和顺序要求！）
        batch_df = pd.DataFrame(current_batch_tasks)
        batch_df = batch_df[["filename", "end_sec", "primary_label", "source"]]
        # embedding 强制作为最右侧的一列
        batch_df["embedding"] = list(embs)
        
        # 5. 写入 Parquet (防内存爆炸)
        table = pa.Table.from_pandas(batch_df)
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(final_output_file, table.schema)
        parquet_writer.write_table(table)
        
        # 清理局部内存
        del batch_audio, x_input, outputs, embs, batch_df, table
        gc.collect()

# 关闭文件写入
if parquet_writer is not None:
    parquet_writer.close()

print(f"\n🎉 大功告成! ")
print(f"⏱️ 总耗时: {(time.time() - global_t0)/60:.2f} 分钟")
print(f"📁 完美的黄金数据集已保存: {final_output_file.name}")

# 测试验证一下成果
test_read = pd.read_parquet(final_output_file)
print(f"📊 最终 Parquet 形状: {test_read.shape}")
print(f"✅ 列名检查: {list(test_read.columns)}")

🔥 开始流式提取并写入...


特征提取进度:   0%|          | 0/540 [00:00<?, ?it/s]


🎉 大功告成! 
⏱️ 总耗时: 116.29 分钟
📁 完美的黄金数据集已保存: teacher_features_label.parquet
📊 最终 Parquet 形状: (34501, 5)
✅ 列名检查: ['filename', 'end_sec', 'primary_label', 'source', 'embedding']


In [7]:
# %% [code]
# ── Cell 6: 构建 C(206,2) 随机组合 Mixup 任务池 ──
import itertools
import random
import pandas as pd
from pathlib import Path

print("正在构建 C(206,2) Mixup 任务清单...")

# 1. 专门读取 train-audio 的 CSV
df_audio_mix = pd.read_csv("/kaggle/input/datasets/waozhang/bc26train-audio/train_audio_argmax_segments.csv")
df_audio_mix["filename"] = df_audio_mix["filepath"].apply(lambda x: Path(x).name)
df_audio_mix["start_sec"] = df_audio_mix["best_start_sec"].astype(float)
df_audio_mix["end_sec"] = df_audio_mix["best_end_sec"].astype(float)

# 2. 按 primary_label 将所有音频分组进入“样本池”
species_groups = df_audio_mix.groupby("primary_label")
species_list = list(species_groups.groups.keys())
n_species = len(species_list)

print(f"📊 共有 {n_species} 个物种参与 Mixup")

# 3. 生成 C(206, 2) 的所有组合
species_pairs = list(itertools.combinations(species_list, 2))
print(f"🧮 组合数 C({n_species}, 2) = {len(species_pairs)} 对")

mixup_tasks =[]

# 4. 对每一对组合，在它们各自的池子里随机抽样一个音频
# random.seed(42) # 可选：为了可复现可以固定随机种子
for sp_A, sp_B in species_pairs:
    # 获取属于该物种的 DataFrame 片段
    pool_A = species_groups.get_group(sp_A)
    pool_B = species_groups.get_group(sp_B)
    
    # 随机取样 1 条
    sample_A = pool_A.sample(n=1).iloc[0]
    sample_B = pool_B.sample(n=1).iloc[0]
    
    # 按照你的要求构建任务字典
    mixup_tasks.append({
        "A_filepath": sample_A["filepath"],
        "A_name": sample_A["filename"],
        "A_start": sample_A["start_sec"],
        "A_end": sample_A["end_sec"],
        "A_label": sp_A,
        
        "B_filepath": sample_B["filepath"],
        "B_name": sample_B["filename"],
        "B_start": sample_B["start_sec"],
        "B_end": sample_B["end_sec"],
        "B_label": sp_B,
        
        "primary_label": f"{sp_A};{sp_B}" # 分号分隔的新标签
    })

print(f"✅ 成功生成 {len(mixup_tasks)} 个 Mixup 任务！准备送入提取器。")

正在构建 C(206,2) Mixup 任务清单...
📊 共有 206 个物种参与 Mixup
🧮 组合数 C(206, 2) = 21115 对
✅ 成功生成 21115 个 Mixup 任务！准备送入提取器。


In [8]:
# %% [code]
# ── Cell 7: 多线程 Mixup 提取与 Parquet 写入 ──
import time
import gc
import concurrent.futures
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import librosa
import soundfile as sf
from tqdm.auto import tqdm

# 如果上文已经定义过，直接复用；如果没有这里作个保险
BATCH_SIZE = 64
NUM_WORKERS = 4
OUTPUT_DIR = Path("/kaggle/working/extracted_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_mixup_file = OUTPUT_DIR / "teacher_mixup_features_label.parquet"
parquet_writer_mix = None

def load_and_cut(filepath, start_sec, end_sec):
    """底层的 5 秒读取工具，供 Mixup 调用"""
    try:
        y, sr = sf.read(str(filepath), dtype="float32", always_2d=False)
        if y.ndim == 2: y = y.mean(axis=1)
        if sr != SR: y = librosa.resample(y, orig_sr=sr, target_sr=SR)
        
        start_frame = int(start_sec * SR)
        end_frame = int(end_sec * SR)
        y_slice = y[start_frame:end_frame]
        
        if len(y_slice) < WINDOW_SAMPLES:
            y_slice = np.pad(y_slice, (0, WINDOW_SAMPLES - len(y_slice)))
        else:
            y_slice = y_slice[:WINDOW_SAMPLES]
        return y_slice
    except Exception as e:
        return np.zeros(WINDOW_SAMPLES, dtype=np.float32)

def process_mixup_task(task):
    """核心：读取A，读取B，使用 Numpy 进行底层 0.5 * A + 0.5 * B 混合"""
    audio_A = load_and_cut(task["A_filepath"], task["A_start"], task["A_end"])
    audio_B = load_and_cut(task["B_filepath"], task["B_start"], task["B_end"])
    
    # 极致性能的原生 NumPy 混合
    mixed_audio = 0.5 * audio_A + 0.5 * audio_B
    return mixed_audio.astype(np.float32)

global_t0 = time.time()
print(f"🔥 开始流式 Mixup 提取并写入: {final_mixup_file.name}")

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    
    next_batch_tasks = mixup_tasks[0:BATCH_SIZE]
    future_audio =[executor.submit(process_mixup_task, t) for t in next_batch_tasks]
    
    for i in tqdm(range(0, len(mixup_tasks), BATCH_SIZE), desc="Mixup 特征提取"):
        current_batch_tasks = next_batch_tasks
        
        # 1. 获取混音波形
        batch_audio =[f.result() for f in future_audio]
        
        # 2. 派发下个批次
        next_start = i + BATCH_SIZE
        if next_start < len(mixup_tasks):
            next_batch_tasks = mixup_tasks[next_start : next_start + BATCH_SIZE]
            future_audio =[executor.submit(process_mixup_task, t) for t in next_batch_tasks]
            
        # 3. ONNX 提取特征
        x_input = np.vstack(batch_audio)
        outputs = ort_session.run([emb_output_name], {input_name: x_input})
        embs = outputs[0]  # (Batch, 1536)
        
        # 4. 组装列数据（剔除 filepath 等中间信息，严格按要求生成）
        clean_batch_data =[]
        for task in current_batch_tasks:
            clean_batch_data.append({
                "A_name": task["A_name"],
                "A_end": task["A_end"],
                "A_label": task["A_label"],
                "B_name": task["B_name"],
                "B_end": task["B_end"],
                "B_label": task["B_label"],
                "primary_label": task["primary_label"]
            })
            
        batch_df = pd.DataFrame(clean_batch_data)
        
        # 5. 将 embedding 强制放到最右侧一列
        batch_df["embedding"] = list(embs)
        
        # 6. 流式写入 Parquet
        table = pa.Table.from_pandas(batch_df)
        if parquet_writer_mix is None:
            parquet_writer_mix = pq.ParquetWriter(final_mixup_file, table.schema)
        parquet_writer_mix.write_table(table)
        
        del batch_audio, x_input, outputs, embs, batch_df, table, clean_batch_data
        gc.collect()

if parquet_writer_mix is not None:
    parquet_writer_mix.close()

print(f"\n🎉 Mixup 数据提取完毕! ")
print(f"⏱️ 总耗时: {(time.time() - global_t0)/60:.2f} 分钟")

# 快速验证一下列的顺序和结果
test_read = pd.read_parquet(final_mixup_file)
print(f"📊 Mixup Parquet 形状: {test_read.shape}")
print(f"✅ 列名检查 (最后一列应为 embedding):")
for col in test_read.columns:
    print(f"  - {col}")

🔥 开始流式 Mixup 提取并写入: teacher_mixup_features_label.parquet


Mixup 特征提取:   0%|          | 0/330 [00:00<?, ?it/s]


🎉 Mixup 数据提取完毕! 
⏱️ 总耗时: 74.63 分钟
📊 Mixup Parquet 形状: (21115, 8)
✅ 列名检查 (最后一列应为 embedding):
  - A_name
  - A_end
  - A_label
  - B_name
  - B_end
  - B_label
  - primary_label
  - embedding


In [9]:
import pandas as pd

# 你的文件路径
file_path = "/kaggle/working/extracted_features/teacher_features_label.parquet"
mixup_file_path = "/kaggle/working/extracted_features/teacher_mixup_features_label.parquet"
# 读取全部（小数据量没事），然后只取前5行
df1 = pd.read_parquet(file_path)
df2 = pd.read_parquet(mixup_file_path)

# 打印列名
print('teacher_features_label信息：')
print("===== 列名 =====")
print(df1.columns.tolist())

# 打印前5行
print("\n===== 数据 =====")
print(df1.head())
print(df1.tail())

# 打印列名
print('teacher_mixup_features_label信息：')
print("===== mixup列名 =====")
print(df2.columns.tolist())

# 打印前5行
print("\n===== mixup数据 =====")
print(df2.head())
print(df2.tail())

teacher_features_label信息：
===== 列名 =====
['filename', 'end_sec', 'primary_label', 'source', 'embedding']

===== 数据 =====
          filename  end_sec primary_label       source  \
0  iNat1114648.ogg     22.5       1161364  train-audio   
1  iNat1216197.ogg     15.0       1161364  train-audio   
2  iNat1264238.ogg      5.0       1161364  train-audio   
3   iNat547199.ogg     12.5       1161364  train-audio   
4   iNat840159.ogg      5.0       1161364  train-audio   

                                           embedding  
0  [0.09220965, -0.1417766, 0.0065071713, 0.01200...  
1  [-0.02021367, -0.15186433, -0.07492009, -0.075...  
2  [0.0025643497, 0.021909162, 0.096098915, 0.261...  
3  [-0.18431926, -0.06863827, -0.06454603, -0.022...  
4  [-0.08742841, 0.030351318, -0.009026574, 0.009...  
                                        filename  end_sec  \
34496  BC2026_Train_0055_S22_20220219_200000.ogg     40.0   
34497  BC2026_Train_0055_S22_20220219_200000.ogg     45.0   
34498  BC2026_Tra

In [10]:
# import pandas as pd

# # 你的文件路径
# file_path = "/kaggle/input/notebooks/waozhang/bc2026-perch-featrue1/extracted_features/train_soundscapes_features.parquet"
# # mixup_file_path = "/kaggle/working/extracted_features/teacher_features_label.parquet"
# # 读取全部（小数据量没事），然后只取前5行
# df = pd.read_parquet(file_path)

# # 打印列名
# print("===== 列名 =====")
# print(df.columns.tolist())

# # 打印前5行
# print("\n===== 数据 =====")
# print(df.head())
# print(df.tail())